In [6]:
import pandas as pd
from pathlib import Path

def analyze_noise_results_per_file(folder_path="03-results"):
    folder = Path(folder_path)
    csv_files = list(folder.glob("*.csv"))
    
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in '{folder_path}'")
        
    expected_categories = ['markup', 'marketing', 'symbols']
    expected_total = 100
    
    # List to store metrics for the final summary dataframe
    summary_data = []
    
    for file_path in csv_files:
        filename = file_path.name
        print(f"\n{'='*50}")
        print(f"📄 Analyzing File: {filename}")
        print(f"{'='*50}")
        
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue
            
        category_counts = df['Noise_Category'].value_counts()
        
        recalls = []
        file_metrics = {'Filename': filename}
        
        print("--- Category Counts & Missing Check ---")
        for cat in expected_categories:
            count = category_counts.get(cat, 0)
            missing = expected_total - count
            
            # True Positive Rate (Recall) for the class
            valid_count = min(count, expected_total)
            recall = valid_count / expected_total
            recalls.append(recall)
            
            # Store for summary
            file_metrics[f'{cat.capitalize()}_Count'] = count
            file_metrics[f'{cat.capitalize()}_Missing'] = missing
            
            print(f"{cat.capitalize():>10}: Present = {count:<4} | Missing = {missing} " + 
                  ("(Perfect!)" if missing == 0 else ""))
            
        # Compute file-level metrics
        balanced_accuracy = sum(recalls) / len(expected_categories)
        avg_confidence = df['Final_Confidence'].mean()
        avg_l_suff = df['L_suff'].mean()
        
        file_metrics['Balanced_Accuracy'] = balanced_accuracy
        file_metrics['Avg_Confidence'] = avg_confidence
        file_metrics['Avg_L_suff'] = avg_l_suff
        summary_data.append(file_metrics)
        
        print("\n--- Final Metrics ---")
        print(f"Balanced Accuracy:        {balanced_accuracy:.2%}")
        print(f"Average Final_Confidence: {avg_confidence:.4f}")
        print(f"Average L_suff:           {avg_l_suff:.4f}")

    # Create and return a summary dataframe for easy viewing later
    summary_df = pd.DataFrame(summary_data)
    
    print(f"\n{'='*50}")
    print(f"✅ Processed {len(summary_df)} files.")
    
    return summary_df

# Execution:
# file_summary_df = analyze_noise_results_per_file("03-results")
# print(file_summary_df)

In [7]:
file_summary_df = analyze_noise_results_per_file("../data/03-results")
print(file_summary_df)


📄 Analyzing File: layer_sufficiency_results_gpt2-small_100_0.9.csv
--- Category Counts & Missing Check ---
    Markup: Present = 8    | Missing = 92 
 Marketing: Present = 9    | Missing = 91 
   Symbols: Present = 13   | Missing = 87 

--- Final Metrics ---
Balanced Accuracy:        10.00%
Average Final_Confidence: 0.1530
Average L_suff:           10.0667

📄 Analyzing File: layer_sufficiency_results_10_0.9.csv
--- Category Counts & Missing Check ---
    Markup: Present = 9    | Missing = 91 
 Marketing: Present = 9    | Missing = 91 
   Symbols: Present = 9    | Missing = 91 

--- Final Metrics ---
Balanced Accuracy:        9.00%
Average Final_Confidence: 0.6360
Average L_suff:           18.0000

📄 Analyzing File: layer_sufficiency_results_gpt2-large_100_0.9.csv
--- Category Counts & Missing Check ---
    Markup: Present = 76   | Missing = 24 
 Marketing: Present = 58   | Missing = 42 
   Symbols: Present = 61   | Missing = 39 

--- Final Metrics ---
Balanced Accuracy:        65.00%
